# Microsoft Foundry Local 
## Mistral7b

<img src="https://github.com/retkowsky/foundry-local/blob/main/foundrylocal.jpg?raw=true">

This notebook demonstrates how to run Mistral 7B Instruct v0.2 fully locally using Microsoft Foundry Local, leveraging an OpenAI‑compatible API for seamless integration with existing applications.
The notebook is designed as a hands‑on, end‑to‑end walkthrough, covering setup, inference, advanced prompting, structured outputs, RAG, and batch processing — all without sending data to the cloud.

- ✅ How to run a modern LLM 100% locally
- ✅ How to reuse existing OpenAI SDK code with Foundry Local
- ✅ How a 7B model with a 32K context window can handle real workloads
- ✅ Practical enterprise use cases: privacy, sovereignty, offline AI

## What is Mistral 7B?
- Mistral 7B is a cutting-edge language model boasting 7.3 billion parameters, which excels in various benchmarks, even surpassing larger models such as Llama 2 13B. 
- It employs advanced methods like Grouped-Query Attention (GQA) to enhance inference speed and Sliding Window Attention (SWA) to effectively handle extensive sequences. 
- Available under the Apache 2.0 license, Mistral 7B can be deployed across multiple platforms, including local infrastructures and major cloud services. Additionally, a unique variant called Mistral 7B Instruct has demonstrated exceptional abilities in task execution, consistently outperforming rivals like Llama 2 13B Chat in certain applications. 
- This adaptability and performance make Mistral 7B a compelling choice for both developers and researchers seeking efficient solutions. Its innovative features and strong results highlight the model's potential impact on natural language processing projects.

https://mistral.ai/news/announcing-mistral-7b


> **Docs:** [Foundry Local Documentation](https://learn.microsoft.com/azure/ai-foundry/foundry-local/)  
> **SDK Reference:** [Python SDK Reference](https://learn.microsoft.com/azure/ai-foundry/foundry-local/reference/reference-sdk?pivots=programming-language-python)

<img src="https://cms.mistral.ai/assets/01c5ceab-a881-4b56-af40-82f0197bece3">
<img src="https://cms.mistral.ai/assets/42b76b3a-e891-461b-b5fd-ed68d7f9dfa6" width=800>

## Author

| Field | Details |
| --- | --- |
| Name | Serge Retkowsky |
| Email | serge.retkowsky@microsoft.com |
| LinkedIn | https://www.linkedin.com/in/serger/ |
| Medium publications | https://medium.com/@sergems18/ |

In [1]:
import datetime
import GPUtil
import json
import matplotlib.pyplot as plt
import os
import platform
import psutil
import requests
import pandas as pd
import sys
import statistics
import time

from collections import Counter
from foundry_local import FoundryLocalManager
from openai import OpenAI

In [2]:
print(f"Python version: {sys.version}")

Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]


In [3]:
print(f"Today is {datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

Today is 25-Feb-2026 17:38:40


In [4]:
print(f"💻 OS: {platform.system()} {platform.release()}")
print(f"- CPU: {platform.processor()}")
print(f"- CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical")

ram = psutil.virtual_memory()
print(f"- RAM total:     {ram.total / (1024**3):.1f} GB")
print(f"- RAM available: {ram.available / (1024**3):.1f} GB")
print(f"- RAM used:      {ram.percent}%")

for part in psutil.disk_partitions():
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"\n💾 Disk [{part.device}] mounted on {part.mountpoint}")
        print(f"- Total: {usage.total / (1024**3):.1f} GB")
        print(f"- Used:  {usage.used / (1024**3):.1f} GB ({usage.percent}%)")
        print(f"- Free:  {usage.free / (1024**3):.1f} GB")
    except PermissionError:
        pass

gpus = GPUtil.getGPUs()

if not gpus:
    print("No GPU detected.")
else:
    for i, gpu in enumerate(gpus):
        print(f"\n🎮 GPU {i} — {gpu.name}")
        print(f"- VRAM Total : {gpu.memoryTotal:,.0f} MB")
        print(
            f"- VRAM Used  : {gpu.memoryUsed:,.0f} MB ({gpu.memoryUtil * 100:.0f}%)"
        )
        print(f"- VRAM Free  : {gpu.memoryFree:,.0f} MB")
        print(f"- GPU Load   : {gpu.load * 100:.0f}%")
        print(f"- Temperature: {gpu.temperature} °C")

💻 OS: Windows 11
- CPU: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
- CPU cores: 8 physical, 16 logical
- RAM total:     63.7 GB
- RAM available: 38.6 GB
- RAM used:      39.4%

💾 Disk [C:\] mounted on C:\
- Total: 951.6 GB
- Used:  878.8 GB (92.3%)
- Free:  72.9 GB

🎮 GPU 0 — NVIDIA T1200 Laptop GPU
- VRAM Total : 4,096 MB
- VRAM Used  : 1,542 MB (38%)
- VRAM Free  : 2,394 MB
- GPU Load   : 38%
- Temperature: 71.0 °C


In [5]:
manager = FoundryLocalManager()
manager.start_service()

print(f"Service running : {manager.is_service_running()}")
print(f"Service URI     : {manager.service_uri}")
print(f"Endpoint (v1)   : {manager.endpoint}")
print(f"Cache location  : {manager.get_cache_location()}")

Service running : True
Service URI     : http://127.0.0.1:55311
Endpoint (v1)   : http://127.0.0.1:55311/v1
Cache location  : C:\models


## Helper

In [6]:
def models_to_df(models):
    """Convert a list of FoundryModelInfo objects into a clean DataFrame."""
    return pd.DataFrame([{
        "alias": m.alias,
        "id": m.id,
        "device": m.device_type.value,
        "provider": m.execution_provider,
        "size_mb": m.file_size_mb,
        "tools": m.supports_tool_calling,
        "license": m.license,
        "task": m.task,
    } for m in models])

## Models

In [7]:
catalog = manager.list_catalog_models()
print(f"Total model variants in catalog = {len(catalog)}")

df_catalog = models_to_df(catalog)
df_catalog

Total model variants in catalog = 80


,alias,id,device,provider,size_mb,tools,license,task
0,phi-4,Phi-4-cuda-gpu:1,GPU,CUDAExecutionProvider,8570,False,MIT,chat-completion
1,phi-4,phi-4-openvino-gpu:1,GPU,OpenVINOExecutionProvider,9046,False,MIT,chat-completion
2,phi-4,Phi-4-generic-gpu:1,GPU,WebGpuExecutionProvider,8570,False,MIT,chat-completion
3,phi-4,Phi-4-generic-cpu:1,CPU,CPUExecutionProvider,10403,False,MIT,chat-completion
4,phi-3.5-mini,Phi-3.5-mini-instruct-cuda-gpu:1,GPU,CUDAExecutionProvider,2181,False,MIT,chat-completion
...,...,...,...,...,...,...,...,...
75,qwen2.5-7b,qwen2.5-7b-instruct-generic-cpu:4,CPU,CPUExecutionProvider,6307,True,apache-2.0,chat-completion
76,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-cuda-gpu:2,GPU,CUDAExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
77,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-generic-cpu:2,CPU,CPUExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
78,gpt-oss-20b,gpt-oss-20b-generic-cpu:1,CPU,CPUExecutionProvider,12552,False,MIT,chat-completion


In [8]:
# Filter by alias keyword
df_catalog[df_catalog["alias"].str.contains("mistral", case=False, na=False)]

,alias,id,device,provider,size_mb,tools,license,task
16,mistral-7b-v0.2,mistralai-Mistral-7B-Instruct-v0-2-cuda-gpu:1,GPU,CUDAExecutionProvider,4075,False,apache-2.0,chat-completion
17,mistral-7b-v0.2,Mistral-7B-Instruct-v0-2-openvino-gpu:1,GPU,OpenVINOExecutionProvider,4374,False,apache-2.0,chat-completion
18,mistral-7b-v0.2,mistralai-Mistral-7B-Instruct-v0-2-generic-gpu:1,GPU,WebGpuExecutionProvider,4167,False,apache-2.0,chat-completion
19,mistral-7b-v0.2,mistralai-Mistral-7B-Instruct-v0-2-generic-cpu:2,CPU,CPUExecutionProvider,4167,False,apache-2.0,chat-completion


In [10]:
# Download first (this may take a few minutes depending on model size)
manager.download_model("mistral-7b-v0.2")

FoundryModelInfo(alias=mistral-7b-v0.2, id=mistralai-Mistral-7B-Instruct-v0-2-cuda-gpu:1, execution_provider=CUDAExecutionProvider, device_type=GPU, file_size=4075 MB, license=apache-2.0)

In [11]:
os.listdir(os.path.join(manager.get_cache_location(), "Microsoft"))

['mistralai-Mistral-7B-Instruct-v0-2-cuda-gpu-1']

In [12]:
# Load a model into memory
model_info = manager.load_model("mistral-7b-v0.2")
print(f"Loaded: {model_info.alias} => {model_info.id}")

Loaded: mistral-7b-v0.2 => mistralai-Mistral-7B-Instruct-v0-2-cuda-gpu:1


In [13]:
info = manager.get_model_info("mistral-7b-v0.2")
print(f"Running on: {info.execution_provider}")
print(f"Device:     {info.device_type}")

Running on: CUDAExecutionProvider
Device:     GPU


In [14]:
cached = manager.list_cached_models()
print(f"Cached models = {len(cached)}")

df_cached = models_to_df(cached)
df_cached

Cached models = 1


,alias,id,device,provider,size_mb,tools,license,task
0,mistral-7b-v0.2,mistralai-Mistral-7B-Instruct-v0-2-cuda-gpu:1,GPU,CUDAExecutionProvider,4075,False,apache-2.0,chat-completion


In [15]:
# Initialize the client
client = OpenAI(base_url=manager.endpoint, api_key=manager.api_key)

print(f"Endpoint: {manager.endpoint}")

Endpoint: http://127.0.0.1:55311/v1


## Basic Chat Completion

Mistral 7B v0.2 is an instruction-tuned model with a 32K context window. Let's start with a simple query.

In [16]:
response = client.chat.completions.create(
    model=model_info.id,
    messages=[{
        "role": "user",
        "content": "What makes Mistral 7B special compared to other 7B parameter models?"
    }],
    temperature=0.7,
    max_tokens=2000)

print("🤖 Mistral 7B says:\n")
print(response.choices[0].message.content)

🤖 Mistral 7B says:

 Mistral 7B is a specific version of the Mistral language model developed by Mistral AI, a French AI company. The "7B" refers to the number of billions of parameters in the model.

Compared to other 7B parameter models, such as Google's T5-XL or Microsoft's Turing-NLG, Mistral 7B has some unique features and capabilities:

1. Multilingual: Mistral 7B is multilingual by design, meaning it can generate text in multiple languages with high accuracy. This is particularly useful in multilingual contexts and for businesses that operate in multiple languages.
2. Fine-tunable: Mistral 7B is fine-tunable, which means it can be fine-tuned on specific tasks or domains. This allows for better performance and customization to specific use cases.
3. Data privacy: Mistral AI emphasizes data privacy and security, and the company offers on-premises deployment options for organizations that require more control over their data.
4. Customizable: Mistral 7B can be customized to meet sp

In [17]:
response.id

'chat.id.115'

In [18]:
response.model

'mistralai-Mistral-7B-Instruct-v0-2-cuda-gpu:1'

## Streaming Responses

Stream tokens as they're generated — essential for real-time applications and better UX.

In [19]:
print("🤖 Streaming response:\n")

stream = client.chat.completions.create(
    model=model_info.id,
    messages=[{
        "role": "user",
        "content": "What is the speed of light?"
    }],
    stream=True,
    max_tokens=400)

token_count = 0
full_response = []

for chunk in stream:
    content = chunk.choices[0].delta.content

    if content is not None:
        print(content, end="", flush=True)
        full_response.append(content)
        token_count += 1

🤖 Streaming response:

 The speed of light is approximately 299,792,458 meters per second (approximately 186,282 miles per second) in a vacuum. This is considered the universal constant and the maximum speed at which all electromagnetic waves, including light, can travel. It is often denoted by the symbol "c". This value has been measured and confirmed through various experiments and observations, and it is a fundamental concept in physics.

## System Prompts & Personas

Use system messages to control Mistral's behavior. The model follows instructions well thanks to its instruction tuning.

In [20]:
def ask_mistral(user_msg, system_msg=None, temperature=0.7, max_tokens=2000):
    """Helper to query Mistral. System prompt is prepended to user message
    since Mistral 7B v0.2 only supports user/assistant roles."""

    # Merge system prompt into the user message
    if system_msg:
        content = f"[Instructions: {system_msg}]\n\n{user_msg}"
    else:
        content = user_msg

    messages = [{"role": "user", "content": content}]

    response = client.chat.completions.create(model=model_info.id,
                                              messages=messages,
                                              temperature=temperature,
                                              max_tokens=max_tokens)
    return response.choices[0].message.content

In [21]:
# Persona: Expert Data Scientist
answer = ask_mistral(
    user_msg="What's the best approach to handle imbalanced datasets?",
    system_msg="You are a senior data scientist with 15 years of experience. "
    "Give practical, actionable advice with concrete examples. Be concise.",
    temperature=0.7)

print(answer)

 Imbalanced datasets are common in real-world data science projects, where one class may have significantly fewer observations than another. Here are some practical and actionable approaches to handle imbalanced datasets:

1. Resampling techniques:
   a. Oversampling: Increase the number of observations in the minority class by duplicating existing instances. Techniques include SMOTE (Synthetic Minority Over-sampling Technique) and ADASYN (Adaptive Synthetic Sampling).
   b. Undersampling: Decrease the number of observations in the majority class by removing instances. Techniques include Random Undersampling and Tomek Links.

Example: In a credit card fraud detection problem, the number of fraudulent transactions is much smaller than the number of non-fraudulent transactions. Use SMOTE to oversample the fraudulent transactions.

2. Cost-sensitive learning:
   Assign different misclassification costs to different classes during the training process. This can be done by setting class wei

In [22]:
answer = ask_mistral(
    user_msg=
    "List 3 popular Python libraries for computer vision with a one-line description each.",
    system_msg='You are a structured data assistant. Always answer in French.'
    'Respond ONLY with valid JSON. No markdown, no explanation, no code fences. '
    'Use the format: [{"name": "...", "description": "..."}]',
    temperature=0.1)

print(answer)

 [{"name": "OpenCV", "description": "Open Source Computer Vision Library. Real-time computer vision."},
 {"name": "Pillow", "description": "Python Imaging Library (PIL) wrapper for OpenCV, OpenCV-Python."},
 {"name": "Scikit-Image", "description": "Scikit-Image: Image processing with scikit-learn."}]


## Multi-Turn Conversations

Build a conversation with Mistral where context accumulates across messages. The model has no memory between API calls — you must pass the full history each time.

In [23]:
class MistralChat:
    """Simple multi-turn chat manager for Mistral."""

    def __init__(self, system_prompt=None):
        self.messages = []
        self.system_prompt = system_prompt

    def chat(self, user_message, max_tokens=2000, temperature=0.7):
        # Prepend system prompt to the first user message
        if self.system_prompt and len(self.messages) == 0:
            user_message = f"{self.system_prompt}\n\n{user_message}"

        self.messages.append({"role": "user", "content": user_message})

        response = client.chat.completions.create(model=model_info.id,
                                                  messages=self.messages,
                                                  max_tokens=max_tokens,
                                                  temperature=temperature)

        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})

        return assistant_msg

    @property
    def turn_count(self):
        return len([m for m in self.messages if m["role"] == "user"])

In [24]:
# Start a multi-turn conversation about machine learning
chat = MistralChat(
    system_prompt="You are a helpful AI tutor. Explain concepts progressively, "
    "building on previous answers. Keep each answer under 200 words.")

turns = [
    "What is a neural network?",
    "How does backpropagation work in that context?",
    "What's the vanishing gradient problem you might encounter?",
    "How do LSTMs solve this?",
]

for question in turns:
    print(f"👤 [{chat.turn_count + 1}] {question}\n")
    answer = chat.chat(question)
    print(f"🤖 {answer}\n")
    print("-" * 100, "\n")

👤 [1] What is a neural network?

🤖  A neural network is a type of machine learning model inspired by the human brain. It's composed of interconnected nodes or "neurons," arranged in layers. The first layer takes in the input data, the last layer outputs the results, and in between are hidden layers where complex computations occur. Each neuron receives input, processes it using an activation function, and sends output to the next neuron. Weights between neurons are adjusted during training to minimize error, enabling the network to learn and improve its performance. Neural networks can learn complex patterns and relationships from data, making them powerful tools for tasks like image recognition, speech recognition, and natural language processing.

---------------------------------------------------------------------------------------------------- 

👤 [2] How does backpropagation work in that context?

🤖  Backpropagation is a supervised learning algorithm used to train neural networks

## Text Summarization

Mistral 7B with its 32K context window can handle long documents. Let's build a summarization pipeline.

In [25]:
def mistral_summarize(text, style="concise", max_words=500, language="English"):
    """
    Summarize a text with Mistral 
    """
    style_prompts = {
        "concise": f"Provide a concise summary in under {max_words} words.",
        "bullet_points": "Summarize as 3-5 bullet points, each under 20 words.",
        "executive": f"Write an executive summary for a business audience, under {max_words} words. "
        "Focus on key takeaways and implications.",
    }

    instruction = (f"You are a professional summarization assistant. "
                   f"{style_prompts.get(style, style_prompts['concise'])} "
                   f"Respond in {language}.")

    response = client.chat.completions.create(
        model=model_info.id,
        messages=[{
            "role": "user",
            "content": f"{instruction}\n\nSummarize this text:\n\n{text}"
        }],
        temperature=0.7,
        max_tokens=200)

    return {
        "summary": response.choices[0].message.content,
        "style": style,
        "language": language,
    }

In [26]:
ARTICLE = """
Mistral AI SAS (French: [mistʁal]) is a French artificial intelligence (AI) 
company, headquartered in Paris. Founded in 2023, it has open-weight large 
language models (LLMs), with both open-source and proprietary AI models.
As of 2025 the company has a valuation of more than US$14 billion.

The company is named after the mistral, a powerful, cold wind in southern 
France, a term which originates from the Occitan language.

Mistral AI was established in April 2023 by three French AI researchers, 
Arthur Mensch, Guillaume Lample and Timothée Lacroix.
Mensch, an expert in advanced AI systems, is a former employee of Google DeepMind; 
Lample and Lacroix, meanwhile, are large-scale AI models specialists who had 
worked for Meta Platforms.
The trio originally met during their studies at École Polytechnique.

Funding:
- In June 2023, the start-up carried out a first fundraising of €105 million 
($117 million) with investors including the American fund Lightspeed Venture 
Partners, Eric Schmidt, Xavier Niel and JCDecaux. The valuation was then 
estimated by the Financial Times at €240 million ($267 million).
- On 10 December 2023, Mistral AI announced that it had raised €385 million 
($428 million) as part of its second fundraising. This round of financing 
involves the Californian fund Andreessen Horowitz, BNP Paribas and the software 
publisher Salesforce.
- By December 2023, it was valued at over $2 billion.
- On 16 April 2024, reporting revealed that Mistral was in talks to raise 
€500 million, a deal that would more than double its current valuation to at 
least €5 billion.
- In June 2024, Mistral AI secured a €600 million ($645 million) funding round, 
increasing its valuation to €5.8 billion ($6.2 billion). Based on valuation, 
as of June 2024, the company was ranked fourth globally in the AI industry, 
and first outside the San Francisco Bay Area.
- In August 2025, the Financial Times reported that Mistral was in talks to 
raise $1 billion at a $10 billion valuation. In September 2025, Bloomberg 
announced that Mistral AI has secured a €2 billion investment valuing it at 
€12 billion ($14 billion).
- This comes after $1.5 billion investment from Dutch company ASML, which owns 
11% of Mistral.

Partnerships:
- On 26 February 2024, Microsoft announced that Mistral's language models would 
be made available on Microsoft's Azure cloud, while the multilingual 
conversational assistant Le Chat would be launched in the style of ChatGPT.
- The partnership also included a financial investment of $16 million by 
Microsoft in Mistral AI.
- In April 2025, Mistral AI announced a €100 million partnership with the 
shipping company CMA CGM.
"""

In [27]:
print(ARTICLE)


Mistral AI SAS (French: [mistʁal]) is a French artificial intelligence (AI) 
company, headquartered in Paris. Founded in 2023, it has open-weight large 
language models (LLMs), with both open-source and proprietary AI models.
As of 2025 the company has a valuation of more than US$14 billion.

The company is named after the mistral, a powerful, cold wind in southern 
France, a term which originates from the Occitan language.

Mistral AI was established in April 2023 by three French AI researchers, 
Arthur Mensch, Guillaume Lample and Timothée Lacroix.
Mensch, an expert in advanced AI systems, is a former employee of Google DeepMind; 
Lample and Lacroix, meanwhile, are large-scale AI models specialists who had 
worked for Meta Platforms.
The trio originally met during their studies at École Polytechnique.

Funding:
- In June 2023, the start-up carried out a first fundraising of €105 million 
($117 million) with investors including the American fund Lightspeed Venture 
Partners, Eric Sch

In [28]:
# Test different styles
for style in ["concise", "bullet_points", "executive"]:
    result = mistral_summarize(ARTICLE, style=style)
    print(f"\n{'-' * 100}")
    print(f"📝 Style: {result['style']}")
    print(f"\n{'-' * 100}\n")
    print(f"{result['summary']}")


----------------------------------------------------------------------------------------------------
📝 Style: concise

----------------------------------------------------------------------------------------------------

 Mistral AI SAS, a French AI company founded in 2023 by Arthur Mensch, Guillaume Lample, and Timothée Lacroix, has seen significant growth and success. The company, named after the powerful wind in southern France, has raised over €1.3 billion ($1.45 billion) in funding from investors including Lightspeed Venture Partners, Eric Schmidt, Xavier Niel, JCDecaux, Andreessen Horowitz, BNP Paribas, Salesforce, and Microsoft. With a current valuation of over $14 billion, Mistral AI is a leading player in the global AI industry, ranking fourth globally and first outside the San Francisco Bay Area.

The company's founders, all French AI experts, met during their studies at École Polytechnique. Mensch, an expert in advanced AI systems, previously worked at Google DeepMind, whil

## Structured Data Extraction

Extract JSON from unstructured text — perfect for ETL pipelines, document processing, and data labeling.

In [29]:
def extract_data(text, schema_hint, temperature=0.5):
    """Extract structured JSON data from unstructured text."""
    instruction = (
        "You are a data extraction engine. "
        "Extract the requested information and return ONLY valid JSON. "
        "No markdown code fences, no explanation, just the raw JSON object. "
        f"Expected schema: {schema_hint}")

    response = client.chat.completions.create(
        model=model_info.id,
        messages=[{
            "role": "user",
            "content": f"{instruction}\n\nText:\n{text}"
        }],
        temperature=temperature,
        max_tokens=2000)

    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

    try:
        return {"data": json.loads(raw), "success": True, "raw": raw}
    except json.JSONDecodeError:
        return {"data": None, "success": False, "raw": raw}

In [30]:
print("Test 1: CV / Resume parsing\n")

r1 = extract_data(
    "Jeanne Martin, ingénieure logiciel senior chez Airbus à Toulouse. "
    "10 ans d'expérience en Python, C++, et Rust. Master en informatique "
    "de l'INSA Toulouse (2014). Parle français, anglais et espagnol.",
    '{"name": str, "title": str, "company": str, "city": str, '
    '"experience_years": int, "skills": [str], "education": str, "languages": [str]}'
)

if r1["success"]:
    print(json.dumps(r1["data"], indent=2, ensure_ascii=False))
else:
    print(f"Parse failed. Raw output:\n{r1['raw']}")

Test 1: CV / Resume parsing

{
  "name": "Jeanne Martin",
  "title": "ingénieure logiciel senior",
  "company": "Airbus",
  "city": "Toulouse",
  "experience_years": 10,
  "skills": [
    "Python",
    "C++",
    "Rust"
  ],
  "education": "Master, INSA Toulouse (2014)",
  "languages": [
    "français",
    "anglais",
    "espagnol"
  ]
}


In [31]:
print("Test 2: Invoice parsing\n")

r2 = extract_data(
    "Invoice #INV-2026-0042 from TechSupplies SARL dated February 15, 2026. "
    "Items: 3x NVIDIA A100 GPU at €9,500 each, 2x Dell PowerEdge R760 server at €4,200 each. "
    "Subtotal: €36,900. VAT (20%): €7,380. Total: €44,280. "
    "Payment due: March 15, 2026. Customer: DataCorp SAS, Paris.",
    '{"invoice_number": str, "vendor": str, "date": str, "items": [{"description": str, '
    '"quantity": int, "unit_price": float}], "subtotal": float, "vat": float, '
    '"total": float, "due_date": str, "customer": str}')

if r2["success"]:
    print(json.dumps(r2["data"], indent=2, ensure_ascii=False))
else:
    print(f"Parse failed. Raw output:\n{r2['raw']}")

Test 2: Invoice parsing

Parse failed. Raw output:
{
"invoice\_number": "INV-2026-0042",
"vendor": "TechSupplies SARL",
"date": "February 15, 2026",
"items": [
{
"description": "NVIDIA A100 GPU",
"quantity": 3,
"unit\_price": 9500.0
},
{
"description": "Dell PowerEdge R760 server",
"quantity": 2,
"unit\_price": 4200.0
}
],
"subtotal": 36900.0,
"vat": 7380.0,
"total": 44280.0,
"due\_date": "March 15, 2026",
"customer": "DataCorp SAS, Paris"
}


In [32]:
print("Test 3: Error log parsing\n")

r3 = extract_data(
    "[2026-02-20 14:32:01 ERROR] ConnectionRefusedError: Cannot connect to "
    "PostgreSQL at db-prod-03.internal:5432. Retry attempt 3/5 failed. "
    "Service: user-auth-service. Pod: auth-pod-7b8c9d. Namespace: production.",
    '{"timestamp": str, "level": str, "error_type": str, "target_host": str, '
    '"port": int, "retry_attempt": str, "service": str, "pod": str, "namespace": str}'
)

if r3["success"]:
    print(json.dumps(r3["data"], indent=2))
else:
    print(f"Parse failed. Raw output:\n{r3['raw']}")

Test 3: Error log parsing

Parse failed. Raw output:
{
"timestamp": "2026-02-20 14:32:01",
"level": "ERROR",
"error\_type": "ConnectionRefusedError",
"target\_host": "db-prod-03.internal",
"port": 5432,
"retry\_attempt": "3/5",
"service": "user-auth-service",
"pod": "auth-pod-7b8c9d",
"namespace": "production"
}


## Code Generation

Mistral 7B can generate and explain code. Let's use it as a local code assistant.

In [33]:
def build_messages(system_msg, user_msg):
    """Build messages compatible with all Foundry Local models."""
    if "mistral" in model_info.id.lower():
        return [{"role": "user", "content": f"{system_msg}\n\n{user_msg}"}]

    return [{
        "role": "system", "content": system_msg
    }, {
        "role": "user", "content": user_msg
    }]

In [34]:
def generate_code(task, language="Python", explain=True):
    """Generate code for a given task."""
    system_msg = (
        f"You are an expert {language} developer. "
        "Write clean, production-ready code with proper error handling. "
        "Keep it concise and well-documented.")

    instruction = f"Write {language} code to: {task}"

    if explain:
        instruction += "\nInclude brief comments explaining key steps."

    response = client.chat.completions.create(model=model_info.id,
                                              messages=build_messages(
                                                  system_msg, instruction),
                                              temperature=0.2,
                                              max_tokens=2000)
    return response.choices[0].message.content

In [35]:
# Generate Python code examples
tasks = [
    "Read a CSV file with pandas, handle missing values, and compute basic statistics",
    "Create a simple REST API endpoint with FastAPI that returns JSON",
    "Implement a binary search algorithm with type hints",
]

for task in tasks:
    print(f"\n{'-'*100}")
    print(f"Task: {task}\n")
    code = generate_code(task)
    print(code)


----------------------------------------------------------------------------------------------------
Task: Read a CSV file with pandas, handle missing values, and compute basic statistics

 ```python # Import required libraries import pandas as pd

# Define function to read CSV file and handle missing values def read_csv_with_missing_values(file_path): try: # Read CSV file into a DataFrame df = pd.read_csv(file_path, na_values=["NaN"], keep_default_na=False, engine='python')

 # Handle missing values by replacing them with a default value, e.g. 0 for numeric columns and "" for string columns. df = df.fillna({"numeric_column": 0, "string_column": ""})

 # Check if there are any missing values in the DataFrame if not df.isnull().values.any().all(): print("No missing values found.") else: print("Missing values found. Replaced with default values.")

 return df

 # Sample usage with a CSV file in the same directory as the script file csv_file = "sample.csv" data = read_csv_with_missing_va

In [36]:
# Code review / explanation

code_to_review = '''
def process(data):
    r = []
    for i in range(len(data)):
        if data[i] != None and data[i] > 0:
            r.append(data[i] ** 2)
    return sorted(r, reverse=True)[:10]
'''

In [37]:
answer = ask_mistral(
    user_msg=
    f"Review this Python code. Identify issues and suggest improvements:\n```python{code_to_review}```",
    system_msg="You are a senior Python code reviewer. "
    "Focus on: correctness, Pythonic style, readability, performance. "
    "Provide a corrected version of the code.",
    temperature=0.7,
    max_tokens=500)

print("Code Review:\n")
print(answer)

Code Review:

 Here are some suggestions for improving the code:

1. Use a more descriptive function name: The function name `process` is not very descriptive. Consider using a name that better reflects what the function does, such as `square_and_take_top_ten`.
2. Use a list comprehension instead of a for loop: You can use a list comprehension to make the code more concise and readable.
3. Use `list filter` instead of checking for `None` and `> 0`: You can use list filter to filter out the elements that are `None` or less than or equal to 0.
4. Use a constant for the number of elements to return: Instead of hardcoding the number of elements to return in the `sorted` function, you can define a constant at the beginning of the function.

Here's the corrected version of the code:

```python
NUM_ELEMENTS = 10

def square_and_take_top_ten(data):
    """
    Returns the top ten squared elements from the given list.
    """
    squared_data = [x ** 2 for x in data if x is not None and x > 0]


## Translation & Multilingual Tasks

Mistral has strong multilingual capabilities, particularly for European languages. Being a French-origin model, it excels at French.

In [38]:
def translate(text, source_lang, target_lang, preserve_tone=True):
    """Translate text between languages."""
    tone_instruction = " Preserve the original tone and register." if preserve_tone else ""

    system_msg = (
        f"You are a professional translator from {source_lang} to {target_lang}. "
        f"Translate accurately and naturally.{tone_instruction} "
        "Output ONLY the translation, nothing else.")

    response = client.chat.completions.create(model=model_info.id,
                                              messages=build_messages(
                                                  system_msg, text),
                                              temperature=0.5,
                                              max_tokens=2000)
    return response.choices[0].message.content

In [39]:
examples = [
    {
        "text":
        "Machine learning models can now run efficiently on consumer hardware, "
        "opening up new possibilities for privacy-preserving AI applications.",
        "from": "English",
        "to": "French"
    },
    {
        "text":
        "L'intelligence artificielle embarquée représente une avancée majeure "
        "pour la souveraineté numérique des entreprises européennes.",
        "from": "French",
        "to": "English"
    },
    {
        "text":
        "Les modèles de langage transforment la manière dont nous interagissons "
        "avec la technologie au quotidien.",
        "from": "French",
        "to": "Spanish"
    },
]

In [40]:
for ex in examples:
    translation = translate(ex["text"], ex["from"], ex["to"])
    print(f"{ex['from']} => {ex['to']}")
    print(f"Original:    {ex['text']}")
    print(f"Translation: {translation}\n")

English => French
Original:    Machine learning models can now run efficiently on consumer hardware, opening up new possibilities for privacy-preserving AI applications.
Translation:  Les modèles d'apprentissage automatisé peuvent maintenant fonctionner efficacement sur matériel d'usage domestique, offrant de nouvelles perspectives pour des applications d'intelligence artificielle respectant la confidentialité.

French => English
Original:    L'intelligence artificielle embarquée représente une avancée majeure pour la souveraineté numérique des entreprises européennes.
Translation:  Embedded artificial intelligence represents a major advance for the digital sovereignty of European businesses.

French => Spanish
Original:    Les modèles de langage transforment la manière dont nous interagissons avec la technologie au quotidien.
Translation:  Los modelos de lenguaje transforman la manera en que interactuamos con la tecnología en el día a día. (The language models change the way we intera

## Batch Processing Pipeline

Process multiple items with progress tracking — useful for labeling datasets, classifying documents, etc.

In [41]:
def batch_classify(items, categories, system_context=""):
    """
    Classify a list of text items into given categories.

    Args:
        items: List of text strings to classify
        categories: List of valid category labels
        system_context: Additional context for classification

    Returns:
        List of dicts with input, label, confidence, and timing
    """
    categories_str = ", ".join(categories)

    system_msg = (
        f"You are a text classifier. Classify the given text into "
        f"exactly one of these categories: {categories_str}. "
        f"{system_context}"
        "Respond with ONLY a JSON object: "
        '{"label": "CATEGORY", "confidence": 0.X, "reason": "brief reason"}')

    results = []
    total = len(items)

    for i, item in enumerate(items):

        response = client.chat.completions.create(model=model_info.id,
                                                  messages=build_messages(
                                                      system_msg, item),
                                                  temperature=0.1,
                                                  max_tokens=100)

        raw = response.choices[0].message.content.strip()
        try:
            if raw.startswith("```"):
                raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            parsed = {
                "label": "PARSE_ERROR",
                "confidence": 0,
                "reason": raw[:80]
            }

        parsed["input"] = item[:60]
        results.append(parsed)

        status = "✅" if parsed.get("label") != "PARSE_ERROR" else "⚠️"
        print(f"  [{i+1}/{total}] {status} {parsed.get('label', '?'):15s} "
              f"({parsed.get('confidence', '?')}) | "
              f"{item[:45]}...")

    return results

In [42]:
# Classify support tickets

tickets = [
    "I can't log in to my account, I keep getting error 403 after password reset",
    "When will the new API v3 be available? We need the batch endpoint.",
    "Your product crashed my entire server! This is unacceptable, I want a refund!",
    "Great product! We've been using it for 6 months and love the performance.",
    "How do I configure SSL certificates for the on-premise deployment?",
    "Billing shows double charge for last month's subscription.",
    "We'd love to see GPU acceleration support in the next release.",
]

categories = [
    "BUG", "FEATURE_REQUEST", "COMPLAINT", "PRAISE", "QUESTION", "BILLING"
]

results = batch_classify(tickets, categories)

  [1/7] ✅ COMPLAINT       (0.9) | I can't log in to my account, I keep getting ...
  [2/7] ✅ QUESTION        (0.9) | When will the new API v3 be available? We nee...
  [3/7] ✅ COMPLAINT       (0.95) | Your product crashed my entire server! This i...
  [4/7] ✅ PRAISE          (0.9) | Great product! We've been using it for 6 mont...
  [5/7] ✅ QUESTION        (0.9) | How do I configure SSL certificates for the o...
  [6/7] ✅ COMPLAINT       (0.9) | Billing shows double charge for last month's ...
  [7/7] ✅ FEATURE_REQUEST (0.9) | We'd love to see GPU acceleration support in ...


In [43]:
label_counts = Counter(r["label"] for r in results)
avg_confidence = sum(r.get("confidence", 0) for r in results) / len(results)

print("Classification Summary\n")
print(f"Avg confidence = {avg_confidence:.2f}")

print(f"\n   Distribution:")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count * 5)
    print(f"   {label:18s} {bar} {count}")

Classification Summary

Avg confidence = 0.91

   Distribution:
   COMPLAINT          ███████████████ 3
   QUESTION           ██████████ 2
   PRAISE             █████ 1
   FEATURE_REQUEST    █████ 1


## Simple RAG (Retrieval-Augmented Generation)

Build a mini knowledge-base Q&A system — retrieve context from local documents and generate answers with Mistral.

In [44]:
class LocalRAG:
    """Minimal RAG pipeline powered by Mistral on Foundry Local."""

    def __init__(self):
        self.chunks = []

    def add_document(self, title, content, chunk_words=200):
        """Add a document, splitting into chunks."""
        words = content.split()
        for i in range(0, len(words), chunk_words):
            chunk = " ".join(words[i:i + chunk_words])
            self.chunks.append({"title": title, "content": chunk})
        print(
            f"  📄 '{title}' => {len(range(0, len(words), chunk_words))} chunk(s)"
        )

    def _retrieve(self, query, top_k=3):
        """Keyword-based retrieval (replace with vector search in production)."""
        query_tokens = set(query.lower().split())
        scored = []
        for chunk in self.chunks:
            tokens = set(chunk["content"].lower().split())
            score = len(query_tokens & tokens)
            if score > 0:
                scored.append((score, chunk))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in scored[:top_k]]

    def ask(self, question, verbose=False):
        """Ask a question against the knowledge base."""
        relevant = self._retrieve(question)

        if verbose:
            print(f"  🔍 Retrieved {len(relevant)} chunks")
            for r in relevant:
                print(f"     • [{r['title']}] {r['content'][:60]}...")

        context = "\n\n".join(
            f"[{r['title']}]: {r['content']}" for r in
            relevant) if relevant else "No relevant information found."

        system_msg = ("Answer questions based ONLY on the provided context. "
                      "If the context doesn't contain the answer, say so. "
                      "Cite which source you used. Be concise and precise.")
        user_msg = f"Context:\n{context}\n\nQuestion: {question}"

        response = client.chat.completions.create(model=model_info.id,
                                                  messages=build_messages(
                                                      system_msg, user_msg),
                                                  temperature=0.2,
                                                  max_tokens=2000)
        return response.choices[0].message.content

In [45]:
# Build knowledge base

rag = LocalRAG()
print("📚 Building knowledge base...\n")

rag.add_document(
    "Foundry Local",
    """Foundry Local is Microsoft's on-device AI inference solution. It runs AI models 
    locally through a CLI, SDK, or REST API. It supports Windows, macOS, and various 
    accelerators including NVIDIA CUDA, AMD GPUs, Intel NPUs, and Qualcomm NPUs. 
    Minimum requirements are 8 GB RAM and 3 GB disk. It provides an OpenAI-compatible 
    API endpoint and uses ONNX Runtime for optimized inference.""")

rag.add_document(
    "Mistral 7B",
    """Mistral 7B is a 7.3 billion parameter language model released by Mistral AI, 
    a French AI company founded in 2023. The model uses grouped-query attention (GQA) 
    and sliding window attention (SWA) for efficient inference. Version 0.2 supports 
    a 32,768 token context window. It outperforms LLaMA 2 13B on many benchmarks 
    despite being smaller. The model is released under the Apache 2.0 license."""
)

rag.add_document(
    "ONNX Runtime",
    """ONNX Runtime is an open-source inference engine optimized for running machine 
    learning models. It supports multiple execution providers: CPU, CUDA for NVIDIA GPUs, 
    DirectML for Windows GPUs, QNN for Qualcomm NPUs, and OpenVINO for Intel hardware. 
    ONNX Runtime enables cross-platform deployment and automatic hardware optimization. 
    Foundry Local uses ONNX Runtime to efficiently run models on diverse hardware."""
)

📚 Building knowledge base...

  📄 'Foundry Local' => 1 chunk(s)
  📄 'Mistral 7B' => 1 chunk(s)
  📄 'ONNX Runtime' => 1 chunk(s)


In [46]:
questions = [
    "What attention mechanisms does Mistral 7B use?",
    "What hardware accelerators does Foundry Local support?",
    "How does ONNX Runtime relate to Foundry Local?",
    "What is the context window size of Mistral 7B v0.2?",
]

In [47]:
for q in questions:
    print(f"👤 Question: {q}\n")
    answer = rag.ask(q, verbose=True)
    print(f"\n🤖 Answer: {answer}\n")
    print("-" * 100, "\n")

👤 Question: What attention mechanisms does Mistral 7B use?

  🔍 Retrieved 1 chunks
     • [Mistral 7B] Mistral 7B is a 7.3 billion parameter language model release...

🤖 Answer:  Mistral 7B uses grouped-query attention (GQA) and sliding window attention (SWA) for efficient inference. (Source: provided context)

---------------------------------------------------------------------------------------------------- 

👤 Question: What hardware accelerators does Foundry Local support?

  🔍 Retrieved 2 chunks
     • [Foundry Local] Foundry Local is Microsoft's on-device AI inference solution...
     • [ONNX Runtime] ONNX Runtime is an open-source inference engine optimized fo...

🤖 Answer:  Foundry Local supports hardware accelerators including NVIDIA CUDA, AMD GPUs, Intel NPUs, and Qualcomm NPUs. (Source: context provided)

---------------------------------------------------------------------------------------------------- 

👤 Question: How does ONNX Runtime relate to Foundry Local?

  🔍 Retr

## 🎯 Summary

You've explored the full capabilities of **Mistral 7B v0.2** running locally on **Foundry Local**:

| Feature | Status |
|---|---|
| Chat Completions (sync + streaming) | ✅ |
| System Prompts & Personas | ✅ |
| Multi-turn Conversations | ✅ |
| Text Summarization (multi-style, multilingual) | ✅ |
| Structured JSON Extraction | ✅ |
| Code Generation & Review | ✅ |
| Translation (multilingual) | ✅ |
| Batch Classification | ✅ |
| RAG Pipeline | ✅ |
| Performance Benchmarking | ✅ |
| Direct REST API | ✅ |
| Token Counting | ✅ |
| Tool / Function Calling | ❌ (use `phi-4-mini` or `qwen2.5-7b`) |

### Key takeaways
- Mistral 7B v0.2 offers strong performance for a 7B model, especially for **European languages**
- The **32K context window** handles long documents well
- All inference runs **100% locally** — no data leaves your machine
- The OpenAI-compatible API makes migration from cloud to local seamless


> Go to the next notebook